# 02 — Retrieval Configuration Benchmark

Progressive addition of retrieval components:
1. Dense-only
2. BM25-only
3. Hybrid (dense + BM25)
4. Hybrid + RRF
5. Hybrid + RRF + MMR
6. Hybrid + RRF + MMR + Rerank

In [ ]:
import subprocess, json, pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

OUTPUT = pathlib.Path("../eval_results/retrieval_benchmark_nb.json")

result = subprocess.run(
    ["python", "-m", "production_rag.evaluation.retrieval_benchmark",
     "--tenant-id", "eval", "--output", str(OUTPUT)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-3000:])

In [ ]:
data = json.loads(OUTPUT.read_text())
df = pd.DataFrame(data).set_index("stage")
df.style.background_gradient(cmap="RdYlGn", subset=["recall_at_5","recall_at_10","mrr","ndcg_at_10"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quality metrics
df[["recall_at_5", "mrr", "ndcg_at_10"]].plot(ax=axes[0], kind="line", marker="o")
axes[0].set_title("Quality vs. retrieval stage")
axes[0].set_xticks(range(len(df)))
axes[0].set_xticklabels(df.index, rotation=30, ha="right")
axes[0].set_ylim(0, 1)

# Latency
df[["avg_latency_ms"]].plot(ax=axes[1], kind="bar", legend=False, color="steelblue")
axes[1].set_title("Average latency (ms)")
axes[1].set_xlabel("")
axes[1].set_xticklabels(df.index, rotation=30, ha="right")

plt.tight_layout()
plt.savefig("../eval_results/retrieval_benchmark.png", dpi=150)
plt.show()